# PHẦN 0 – CÀI ĐẶT & IMPORT

In [ ]:
# Cài đặt thư viện
!pip install pymc arviz scikit-learn

In [ ]:
# Import cơ bản
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import (
    accuracy_score, f1_score, log_loss, roc_auc_score
)
from sklearn.metrics import confusion_matrix, classification_report
from sklearn.ensemble import IsolationForest
from sklearn.neighbors import LocalOutlierFactor

import pymc as pm
import pytensor.tensor as pt
import arviz as az

sns.set(style="whitegrid")
plt.rcParams["figure.figsize"] = (8, 5)
plt.rcParams["font.size"] = 12


# PHẦN 1 – LOAD DATA & CLEANING (IMPUTE AGE)

Cơ sở dữ liệu này chứa 34 đặc trưng, trong đó 33 đặc trưng có giá trị tuyến tính và một đặc trưng có giá trị định danh.

Thuộc tính lớp là cột cuối cùng. Số lượng mẫu: 366, Số lượng đặc trưng: 34. Giá trị đặc trưng bị thiếu: 8 (trong đặc trưng Tuổi). Được phân biệt bằng dấu '?'.

### Thông tin về giá trị đặc trưng
Trong tập dữ liệu được xây dựng cho lĩnh vực này, đặc trưng tiền sử gia đình có giá trị là 1 nếu bất kỳ bệnh nào trong số này đã được quan sát thấy trong gia đình, và 0 nếu ngược lại. Đặc trưng tuổi chỉ đơn giản thể hiện tuổi của bệnh nhân.

Mọi đặc trưng lâm sàng và mô bệnh học khác được gán một mức độ trong khoảng từ 0 đến 3. Ở đây, 0 biểu thị đặc trưng không có, 3 biểu thị mức độ cao nhất có thể, và 1, 2 biểu thị các giá trị trung gian tương đối.

In [ ]:
# Upload & đọc file dermatology.csv
from google.colab import files
import io
import pandas as pd

uploaded = files.upload()
filename = list(uploaded.keys())[0]

df = pd.read_csv(
    io.BytesIO(uploaded[filename]),
    na_values="?"
)
print(df.shape)
print(df.dtypes)
print("\nTrước khi impute age:")
print(df.isna().sum())
df.head()


In [ ]:
# Kiểm tra missing & Impute age bằng median
median_age = df["age"].median()
df["age"].fillna(median_age, inplace=True)

# Đổi class từ 1–6 -> 0–5 cho mô hình
df["class"] = df["class"].astype(int) - 1

print("\nSau khi impute age:")
print(df.isna().sum())

# PHẦN 2 – EDA CƠ BẢN

## Phân bố lớp (class imbalance)

In [ ]:
# Phân bố số lượng ca bệnh theo lớp

class_counts = df["class"].value_counts().sort_index()

plt.figure(figsize=(8, 5))
ax = sns.barplot(
    x=class_counts.index + 1,
    y=class_counts.values,
    color="steelblue"
)

plt.xlabel("Lớp bệnh (1–6)")
plt.ylabel("Số lượng ca bệnh")
plt.title("Phân bố số lượng ca bệnh (Class Imbalance)")

for i, value in enumerate(class_counts.values):
    ax.text(
        i,
        value + 2,
        str(value),
        ha='center',
        va='bottom',
        fontsize=11,
        fontweight="bold"
    )

plt.tight_layout()
plt.show()


Biểu đồ cho thấy dữ liệu không cân bằng tuyệt đối giữa 6 lớp bệnh. Lớp 1 có số ca nhiều nhất (112 ca), tiếp theo là lớp 3, 5 và 2; trong khi lớp 6 chỉ có 20 ca. Điều này hàm ý các mô hình phân loại cần được đánh giá bằng các thước đo nhạy với mất cân bằng lớp (như Macro F1, AUC) thay vì chỉ dựa trên Accuracy, đồng thời lớp hiếm (lớp 6) dễ bị dự đoán sai nếu mô hình thiên lệch về các lớp phổ biến.

## Phân bố của 8 đặc trưng

In [ ]:
#  Barplot theo lớp bệnh + hiển thị giá trị trung bình

features_to_plot = [
    "erythema",
    "scaling",
    "definite_borders",
    "itching",
    "koebner_phenomenon",
    "polygonal_papules",
    "knee_and_elbow_involvement",
    "family_history"
]

for feat in features_to_plot:
    # Tính giá trị trung bình theo từng lớp bệnh
    mean_by_class = df.groupby("class")[feat].mean().sort_index()

    plt.figure(figsize=(7, 4))
    ax = sns.barplot(
        x=mean_by_class.index + 1,
        y=mean_by_class.values,
        palette="viridis"
    )

    # Ghi giá trị trung bình lên trên mỗi cột
    for i, value in enumerate(mean_by_class.values):
        ax.text(
            i, value + 0.05,       # vị trí (x, y)
            f"{value:.2f}",        # format số
            ha='center', va='bottom',
            fontsize=10, fontweight='bold'
        )

    plt.xlabel("Lớp bệnh (1–6)")
    plt.ylabel(f"Giá trị trung bình {feat}")
    plt.title(f"Trung bình {feat} theo 6 lớp bệnh")
    plt.ylim(0, 3.2)  # để có khoảng trống đặt chữ

    plt.show()


Các biểu đồ này so sánh giá trị trung bình của từng đặc trưng lâm sàng giữa 6 lớp bệnh. Nhìn chung, mỗi đặc trưng có “mẫu hình” khác nhau theo lớp, giúp gợi ý vai trò phân biệt của chúng trong chẩn đoán:

Những đặc trưng có trung bình cao nổi bật ở một vài lớp cụ thể có khả năng là dấu hiệu đặc trưng cho lớp bệnh đó.

Ngược lại, những đặc trưng có giá trị thấp và tương đối đồng đều giữa các lớp ít có sức phân biệt hơn và có thể đóng vai trò hỗ trợ.

In [ ]:
# Phân bố 8 đặc trưng theo thang điểm 0–3

features_to_plot = [
    "erythema",
    "scaling",
    "definite_borders",
    "itching",
    "koebner_phenomenon",
    "polygonal_papules",
    "knee_and_elbow_involvement",
    "family_history"
]

for feat in features_to_plot:
    plt.figure(figsize=(7, 4))
    ax = sns.countplot(x=feat, data=df, palette="viridis")

    mean_val = df[feat].mean()

    for p in ax.patches:
        count = int(p.get_height())
        x = p.get_x() + p.get_width() / 2
        ax.annotate(f"{count}", (x, p.get_height() + 1), ha='center', fontsize=10)

    plt.xlabel(f"Điểm {feat} (0–3)")
    plt.ylabel("Số lượng bản ghi")
    plt.title(f"Phân bố điểm đặc trưng: {feat}")

    plt.xticks([0, 1, 2, 3])
    plt.tight_layout()
    plt.show()


Biểu đồ thể hiện phân bố tần suất điểm 0–3 của đặc trưng <feature> trên toàn bộ tập dữ liệu. Ta thấy phần lớn bệnh nhân tập trung ở các mức 1–2, trong khi mức 0 và 3 chiếm tỉ lệ nhỏ hơn. Điều này cho thấy đa số ca bệnh có biểu hiện <feature> ở mức độ trung gian, còn mức 3 là tương đối hiếm – nhiều khả năng liên quan tới các thể bệnh nặng hoặc điển hình hơn.

## Phân bố theo đặc trưng Age

In [ ]:
# Phân bố tuổi sau khi impute
plt.figure(figsize=(10, 6))

for c in sorted(df["class"].unique()):
    subset = df[df["class"] == c]
    sns.kdeplot(subset["age"], fill=True, alpha=0.2, linewidth=2, label=f"Class {c}")

plt.xlabel("Age")
plt.ylabel("Density")
plt.title("Phân bố độ tuổi theo 6 loại bệnh (KDE Density)")
plt.legend(title="Class")
plt.tight_layout()
plt.show()


Biểu đồ KDE mô tả phân bố tuổi của bệnh nhân trong từng lớp bệnh. Các đường mật độ cho thấy:

Một số lớp bệnh tập trung ở độ tuổi trung niên (khoảng 40–60 tuổi), trong khi các lớp khác có phân bố trẻ hơn.

Các đường mật độ chồng lấn một phần, nghĩa là tuổi không phải yếu tố duy nhất để phân loại, nhưng vẫn cung cấp thông tin bổ trợ: một số lớp có xu hướng “già” hơn hoặc “trẻ” hơn so với những lớp khác.
Điều này hỗ trợ ý nghĩa lâm sàng: mô hình cần kết hợp tuổi với các đặc trưng mô học và lâm sàng khác thay vì dùng riêng lẻ.

In [ ]:
plt.figure(figsize=(8, 5))

ax = sns.boxplot(
    x="class",
    y="age",
    data=df,
    palette="tab10",
    showfliers=True
)
ax.set_xticklabels([1, 2, 3, 4, 5, 6])

plt.xlabel("Class (1–6)")
plt.ylabel("Age")
plt.title("Phân bố tuổi theo 6 loại bệnh")

plt.tight_layout()
plt.show()

Nhìn chung:

Trung vị tuổi giữa các lớp nằm trong khoảng tương đối gần nhau, nhưng độ rộng hộp và số lượng outlier khác nhau, cho thấy độ biến thiên tuổi giữa các nhóm bệnh không đồng nhất.

Một số lớp có nhiều điểm nằm ngoài râu boxplot (outlier tuổi), phản ánh sự hiện diện của các trường hợp rất trẻ hoặc rất lớn tuổi so với mặt bằng chung.
Điều này nhấn mạnh rằng tuổi là biến có độ phân tán rộng và có thể góp phần tạo ra các quan sát “khác thường” trong dữ liệu.

In [ ]:
# ---- Các đặc trưng cần vẽ ----
features_to_show = [
    "erythema",
    "scaling",
    "definite_borders",
    "itching",
    "koebner_phenomenon",      # đặc trưng có outlier
    "polygonal_papules",
    "knee_and_elbow_involvement",
    "family_history"
]

# ---- Lớp 2 trong PAPER = class == 1 trong DataFrame của m ----
df_class2 = df[df["class"] == 1][features_to_show]

# ---- Đổi giá trị '2' trong koebner thành string đỏ ----
styled_df = df_class2.copy().astype(str)
styled_df.loc[df_class2["koebner_phenomenon"] == 2, "koebner_phenomenon"] = "2"

# ---- Render bảng như Table 1 ----
fig, ax = plt.subplots(figsize=(14, 12))

ax.axis('off')
tbl = ax.table(
    cellText=styled_df.values,
    colLabels=styled_df.columns,
    loc='center',
    cellLoc='center'
)

# ---- Highlight màu đỏ cho outlier ----
for i in range(len(styled_df)):
    if styled_df.iloc[i]["koebner_phenomenon"] == "2":
        # cột thứ 5 (index 4)
        tbl[(i+1, 4)].set_facecolor("#ffcccc")
        tbl[(i+1, 4)].set_text_props(color="red", weight="bold")

plt.tight_layout()
plt.show()


Bảng này liệt kê chi tiết một số ca bệnh thuộc lớp 2 cùng giá trị của 8 đặc trưng được lựa chọn. Ô màu đỏ ở cột koebner_phenomenon đánh dấu trường hợp có điểm bằng 2 – đây là các quan sát ít gặp, khác biệt so với đa số còn lại. Việc highlight những điểm bất thường như vậy giúp dễ dàng nhận diện các trường hợp có khả năng là ngoại lai về mặt lâm sàng, từ đó hỗ trợ bước phân tích outlier bằng thuật toán LOF ở phần sau.

# PHẦN 3 – TIỆN ÍCH CHUNG: SPLIT, SCALE, METRICS

In [ ]:
# Hàm tiện ích để chia train/test và scale ===

def prepare_train_test(df, test_size=0.33, random_state=42):
    X = df.drop(columns=["class"])
    y = df["class"].values

    X_train, X_test, y_train, y_test = train_test_split(
        X, y,
        test_size=test_size,
        stratify=y,
        random_state=random_state
    )

    scaler = StandardScaler()
    X_train_scaled = scaler.fit_transform(X_train)
    X_test_scaled = scaler.transform(X_test)

    return X_train_scaled, X_test_scaled, y_train, y_test, scaler


In [ ]:
#  Hàm tính metrics (Accuracy, Macro-F1, Log-loss, ROC-AUC) ===

def evaluate_multiclass(y_true, proba_pred):
    """
    y_true: (n_samples,)
    proba_pred: (n_samples, n_classes) - xác suất dự đoán
    """
    y_pred = proba_pred.argmax(axis=1)
    acc = accuracy_score(y_true, y_pred)
    f1 = f1_score(y_true, y_pred, average="macro")
    ll = log_loss(y_true, proba_pred)

    try:
        auc = roc_auc_score(
            y_true,
            proba_pred,
            multi_class="ovr",
            average="macro"
        )
    except ValueError:
        auc = np.nan

    return {
        "accuracy": acc,
        "macro_f1": f1,
        "log_loss": ll,
        "auc_ovr": auc
    }


# PHẦN 4 – PHÁT HIỆN OUTLIER (LOF) + TẠO DATASET 70–100%


## 1. Định nghĩa Cơ bản về Giá trị Ngoại lai (Outlier)

Giá trị ngoại lai là một **quan sát (bản ghi)** hoặc **một điểm dữ liệu** nằm ở khoảng cách đáng kể so với các quan sát khác (dữ liệu bình thường) trong cùng một tập hợp.

**Nói cách khác:** Đó là dữ liệu không phù hợp với xu hướng chung của bộ dữ liệu.

### 2. Nguyên nhân Xuất hiện Ngoại lai

Ngoại lai có thể đến từ nhiều nguồn khác nhau. Trong y khoa và dữ liệu của bạn, chúng thường do:

* **Lỗi Nhập liệu (Data Entry Errors):** Sai sót khi bác sĩ hoặc nhân viên nhập liệu ghi lại các giá trị (ví dụ: ghi 30 tuổi thành 300 tuổi).
* **Lỗi Thiết bị/Đo lường:** Sai số từ phòng thí nghiệm hoặc quy trình sinh thiết.
* **Hiện tượng Tự nhiên (True Outlier):** Các trường hợp y tế hiếm gặp hoặc bất thường về mặt lâm sàng (**những ca bệnh hợp lệ nhưng không điển hình**).

### 3. Tác động của Ngoại lai lên Mô hình Học máy

Ngoại lai là vấn đề lớn nhất đối với các mô hình ML cổ điển (như Gaussian Naive Bayes, Linear Regression) vì chúng:

* **Gây Nhiễu (Bias):** Kéo đường phân loại/hồi quy ra xa khỏi dữ liệu chính, làm giảm khả năng tổng quát hóa của mô hình.
* **Giảm Độ Chính xác:** Các mô hình sẽ cố gắng phân loại các điểm ngoại lai, dẫn đến việc tạo ra các ranh giới phức tạp và dự đoán sai.


**Mục đích của việc Xóa/Xử lý Ngoại lai:**

* **Tác giả (Mode-Ratio):** Loại bỏ outlier để làm sạch dữ liệu và **đạt kết quả $100\%$ Accuracy** (ưu tiên hiệu suất tuyệt đối).
* **Mô hình Bayesian của em - LOF:** Sử dụng các phương pháp thống kê vững vàng (robust) để **giảm ảnh hưởng của outlier** mà không cần phải xóa chúng, ưu tiên tính vững vàng và sự ổn định của mô hình.

##  Cách Hoạt động của Thuật toán Local Outlier Factor (LOF)

**LOF** là một "cảnh sát mật độ" cục bộ. Nó xác định ngoại lai (outlier) bằng cách trả lời câu hỏi: **"Điểm dữ liệu này có đông đúc như hàng xóm của nó không?"**

### 1. Ý tưởng Cốt lõi: Cụm Dữ liệu và Khoảng cách

* **Dữ liệu Gốc:** Dữ liệu của bạn nằm trong một không gian 34 chiều (34 đặc trưng). Hầu hết các ca bệnh bình thường sẽ tập trung lại thành các **cụm dày đặc** (ví dụ: cụm Psoriasis, cụm Lichen Planus).
* **Điểm Ngoại lai:** Điểm ngoại lai là những ca bệnh nằm **rời rạc** hoặc ở **xa rìa** các cụm đó.

### 2. Quy trình Tính toán (Cảnh sát Mật độ)

Để tính điểm bất thường cho một điểm dữ liệu ($P$), LOF thực hiện 3 bước:

#### A. Xác định Hàng xóm ($k$)
LOF trước hết chọn ra $k$ điểm dữ liệu gần nhất xung quanh $P$ (ví dụ: $k=5$ láng giềng gần nhất).

#### B. Đo Mật độ Cục bộ (LRD)
* **Mật độ Cao:** Nếu $P$ nằm sát $k$ láng giềng của nó, chứng tỏ khu vực này **đông đúc** (mật độ cao).
* **Mật độ Thấp:** Nếu $P$ nằm rất xa $k$ láng giềng của nó, chứng tỏ khu vực này **thưa thớt** (mật độ thấp).

#### C. Tính Hệ số Ngoại lai (LOF Score)
LOF sau đó **so sánh mật độ** của điểm $P$ với mật độ **trung bình** của $k$ láng giềng đó.

* **Nếu $\text{LOF} \approx 1$:** Mật độ của $P$ **ngang bằng** mật độ của hàng xóm $\to$ **Điểm Bình thường.**
* **Nếu $\text{LOF} \gg 1$:** Mật độ của $P$ **thấp hơn nhiều** so với hàng xóm $\to$ **Outlier (Ngoại lai).** (Nó đang cố gắng lẻn ra khỏi đám đông.)

### 3. Ra Quyết định Lọc

Sau khi tính toán điểm LOF cho tất cả các ca bệnh, bạn chỉ cần sắp xếp chúng theo thứ tự giảm dần và **cắt bỏ** những ca bệnh có điểm LOF cao nhất theo tỷ lệ mong muốn ($10\%, 20\%, 30\%$).

**Tóm lại:** LOF xác định outlier bằng cách đo lường mức độ cô lập của điểm đó so với khu vực lân cận của nó trong không gian đa chiều, làm cho nó trở thành một công cụ thống kê mạnh mẽ và khách quan.

In [ ]:
# Hàm đánh dấu outlier LOF

def compute_outlier_scores(X):
    lof = LocalOutlierFactor(n_neighbors=20, novelty=False)
    lof.fit(X)
    scores = -lof.negative_outlier_factor_
    return scores

def filter_by_threshold(df, scores, retain_ratio):

    n = len(df)
    k = int(np.floor(retain_ratio * n))
    idx_sorted = np.argsort(scores)
    keep_idx = idx_sorted[:k]

    return df.iloc[keep_idx].copy()


In [ ]:
# Tạo các bộ dữ liệu cho LOF với 70–100%
X_all = df.drop(columns=["class"]).values

scores_lof = compute_outlier_scores(X_all)

retain_levels = [1.0, 0.9, 0.8, 0.7]

datasets_lof = {}

for r in retain_levels:
    datasets_lof[r] = filter_by_threshold(df, scores_lof, retain_ratio=r)

for r in retain_levels:
    print(f"LOF - retain {int(r*100)}%: {len(datasets_lof[r])} samples")


In [ ]:
# VẼ HISTOGRAM LOF + 4 NGƯỠNG retain (100–90–80–70%)

import numpy as np
import matplotlib.pyplot as plt

# LOF scores (đã tính từ phần LOF ban đầu)
lof_scores = scores_lof

# Các ngưỡng retain mà bài của m đang dùng
retain_levels = [1.0, 0.9, 0.8, 0.7]
thresholds = {}

for r in retain_levels:
    k = int(len(lof_scores) * r)
    sorted_scores = np.sort(lof_scores)
    thresholds[r] = sorted_scores[k - 1]

# ==== Plot ====
plt.figure(figsize=(10, 6))
plt.hist(lof_scores, bins=50, color="steelblue", edgecolor="black", alpha=0.7)

# Vẽ 4 đường threshold
colors = {1.0: "green", 0.9: "red", 0.8: "orange", 0.7: "purple"}

for r in retain_levels:
    plt.axvline(thresholds[r], color=colors[r], linestyle="--", linewidth=2,
                label=f"Threshold retain {int(r*100)}% → {thresholds[r]:.3f}")

plt.xlabel("LOF Score", fontsize=12)
plt.ylabel("Frequency", fontsize=12)
plt.title("Distribution of LOF Scores with Outlier Thresholds (100%–90%–80%–70%)", fontsize=14)
plt.legend()
plt.grid(alpha=0.25)
plt.show()

Biểu đồ mô tả phân bố điểm LOF của toàn bộ 366 ca bệnh. Phần lớn các mẫu có LOF dao động quanh 1.0–1.08, thể hiện các ca nằm trong vùng mật độ “bình thường” của không gian đặc trưng. Các điểm có LOF cao hơn (bên phải đồ thị) là những ca có mật độ lân cận thấp hơn hẳn xung quanh – ứng với các quan sát bất thường hoặc tiềm năng bị gán sai nhãn.
Bốn đường thẳng đứng tương ứng với các ngưỡng cắt để giữ lại 100%, 90%, 80% và 70% dữ liệu. Mỗi ngưỡng được xác định bằng phân vị của phân bố LOF (ví dụ retain 90% nghĩa là cắt bỏ 10% ca có LOF cao nhất). Cách chọn ngưỡng dựa trên phân bố thực tế giúp kiểm soát mức độ làm sạch nhiễu một cách có hệ thống, thay vì chọn tỉ lệ loại bỏ một cách cảm tính.

# PHẦN 5 – MÔ HÌNH BAYESIAN MULTINOMIAL LOGISTIC

### METROPOLIS–HASTINGS

In [ ]:
def softmax_np(z):

    z = z - z.max(axis=1, keepdims=True)
    exp_z = np.exp(z)
    return exp_z / exp_z.sum(axis=1, keepdims=True)


def log_likelihood_multilogit(beta_flat, X, y, K):
    N, D = X.shape
    beta = beta_flat.reshape(D, K)

    logits = X @ beta
    probs = softmax_np(logits)

    eps = 1e-12
    return np.sum(np.log(probs[np.arange(N), y] + eps))


def log_prior(beta_flat, sigma=1.0):

    return -0.5 * np.sum((beta_flat / sigma) ** 2) \
           - beta_flat.size * np.log(sigma * np.sqrt(2 * np.pi))


def log_posterior(beta_flat, X, y, K, sigma=1.0):
    return log_prior(beta_flat, sigma) + log_likelihood_multilogit(beta_flat, X, y, K)


# === Metropolis sampler (có in acceptance rate, KHÔNG đổi interface) ===

def mh_sampler_multilogit(
    X, y, K,
    n_samples=6000,
    step=0.05,
    burn_in=1000,
    sigma_prior=1.0,
    random_state=42,
):
    rng = np.random.default_rng(random_state)
    N, D = X.shape

    # Khởi tạo beta_flat và log posterior hiện tại
    beta_flat = np.zeros(D * K)
    current_lp = log_posterior(beta_flat, X, y, K, sigma_prior)

    samples = []
    accepts = 0  # đếm số bước được chấp nhận

    for i in range(n_samples):
        # Đề xuất beta mới
        proposal = beta_flat + rng.normal(0, step, size=beta_flat.shape)
        proposal_lp = log_posterior(proposal, X, y, K, sigma_prior)

        acc_ratio = np.exp(proposal_lp - current_lp)

        # Metropolis step
        if rng.uniform() < min(1.0, acc_ratio):
            beta_flat = proposal
            current_lp = proposal_lp
            accepts += 1

        samples.append(beta_flat.copy())

    # Tính & in acceptance rate cho lần chạy này
    acceptance_rate = accepts / n_samples
    print(f"Metropolis – acceptance rate: {acceptance_rate:.3f}")

    # Giữ nguyên phần xử lý burn-in + reshape như cũ
    samples = np.array(samples)
    samples = samples[burn_in:]              # bỏ burn-in
    S = samples.shape[0]
    return samples.reshape(S, D, K)



In [ ]:
retain_levels = [1.0, 0.9, 0.8, 0.7]

mh_results = []
mh_conf_mats = {}

for r in retain_levels:
    print(f" METROPOLIS – LOF giữ lại {int(r*100)}% dữ liệu")

    df_r = datasets_lof[r]
    n_total = len(df_r)

    X_train, X_test, y_train, y_test, scaler = prepare_train_test(df_r)
    n_train, n_test = len(X_train), len(X_test)
    print(f"Số mẫu tổng: {n_total} | Train: {n_train} | Test: {n_test}")

    K = len(np.unique(y_train))
    mh_samples = mh_sampler_multilogit(
        X_train, y_train, K,
        n_samples=1500,
        step=0.02,
        burn_in=800,
        sigma_prior=1.0,
        random_state=42
    )

    proba_mh = predict_multilogit_mh(mh_samples, X_test)

    metrics = evaluate_multiclass(y_test, proba_mh)

    y_pred_mh = proba_mh.argmax(axis=1)
    cm_mh = confusion_matrix(y_test, y_pred_mh)
    mh_conf_mats[r] = cm_mh

    mh_results.append({
        "retain_ratio": r,
        "retain_percent": int(r * 100),
        "n_total": n_total,
        "n_train": n_train,
        "n_test": n_test,
        "accuracy_mh": metrics["accuracy"],
        "macro_f1_mh": metrics["macro_f1"],
        "log_loss_mh": metrics["log_loss"],
        "auc_ovr_mh": float(metrics["auc_ovr"]),
    })

mh_results_df = (
    pd.DataFrame(mh_results)
    .sort_values("retain_ratio", ascending=False)
    .reset_index(drop=True)
)
print("\n KẾT QUẢ METROPOLIS")
display(mh_results_df)


## NUTS (pymc)

In [ ]:
def fit_bayes_multilogit_nuts(
    X_train, y_train,
    draws=800, tune=800,
    target_accept=0.9,
    random_seed=42,
    chains=2
):
    import pymc as pm
    import pytensor.tensor as pt
    import numpy as np

    X_train = np.asarray(X_train, dtype="float64")
    y_train = np.asarray(y_train, dtype="int64")

    N, D = X_train.shape
    K = len(np.unique(y_train))

    with pm.Model() as model:
        # Priors
        alpha = pm.Normal("alpha", mu=0, sigma=1, shape=K)
        beta  = pm.Normal("beta",  mu=0, sigma=1, shape=(D, K))

        # Linear predictor
        logits = alpha + pt.dot(X_train, beta)   # (N, K)

        # Likelihood: dùng logit_p trong PyMC5
        y_obs = pm.Categorical("y_obs", logit_p=logits, observed=y_train)

        # Tạo step NUTS ở đây, KHÔNG truyền 'nuts=' vào pm.sample
        step = pm.NUTS(target_accept=target_accept)

        idata = pm.sample(
            draws=draws,
            tune=tune,
            step=step,
            init="auto",
            chains=chains,
            random_seed=random_seed,
            progressbar=True,
        )

    return model, idata

def predict_bayes_multilogit_mean(idata, X_test):
    X_test = np.asarray(X_test)

    alpha_samples = idata.posterior["alpha"].values.reshape(-1, idata.posterior["alpha"].shape[-1])
    beta_samples = idata.posterior["beta"].values.reshape(-1, idata.posterior["beta"].shape[-2], idata.posterior["beta"].shape[-1])

    S, K = alpha_samples.shape
    N, D = X_test.shape
    logits_samples = alpha_samples[:, None, :] + np.einsum("nd,sdk->snk", X_test, beta_samples)

    logits_2d = logits_samples.reshape(S * N, K)
    probs_2d = softmax_np(logits_2d)
    probs_samples = probs_2d.reshape(S, N, K)

    proba_mean = probs_samples.mean(axis=0)

    return proba_mean

In [ ]:
retain_levels = [1.0, 0.9, 0.8, 0.7]

nuts_results = []
nuts_conf_mats = {}

for r in retain_levels:
    print("\n==============================")
    print(f" NUTS – LOF giữ lại {int(r*100)}% dữ liệu")
    print("==============================")

    df_r = datasets_lof[r]
    n_total = len(df_r)

    # 1. Chia train/test + scale
    X_train, X_test, y_train, y_test, scaler = prepare_train_test(df_r)
    n_train, n_test = len(X_train), len(X_test)
    print(f"Số mẫu tổng: {n_total} | Train: {n_train} | Test: {n_test}")

    # 2. Fit model NUTS
    model_nuts, idata_nuts = fit_bayes_multilogit_nuts(
        X_train, y_train,
        draws=1000,
        tune=1000,
        target_accept=0.9,
        random_seed=42,
        chains=2,
    )

    # 3. Dự đoán
    proba_nuts = predict_bayes_multilogit_mean(idata_nuts, X_test)

    # 4. Tính metrics
    metrics = evaluate_multiclass(y_test, proba_nuts)

    # 5. Lưu confusion matrix
    y_pred_nuts = proba_nuts.argmax(axis=1)
    cm_nuts = confusion_matrix(y_test, y_pred_nuts)
    nuts_conf_mats[r] = cm_nuts

    nuts_results.append({
        "retain_ratio": r,
        "retain_percent": int(r * 100),
        "n_total": n_total,
        "n_train": n_train,
        "n_test": n_test,
        "accuracy_nuts": metrics["accuracy"],
        "macro_f1_nuts": metrics["macro_f1"],
        "log_loss_nuts": metrics["log_loss"],
        "auc_ovr_nuts": float(metrics["auc_ovr"]),
    })

nuts_results_df = (
    pd.DataFrame(nuts_results)
    .sort_values("retain_ratio", ascending=False)
    .reset_index(drop=True)
)
print("\n KẾT QUẢ NUTS ")
display(nuts_results_df)


### Bảng so sánh hiệu suất 2 mô hình

In [ ]:
# so sánh Metropolis vs NUTS trên 4 mức LOF

compare_single_df = mh_results_df.merge(
    nuts_results_df,
    on=["retain_ratio", "retain_percent", "n_total", "n_train", "n_test"],
)
print("\n BẢNG SO SÁNH METROPOLIS vs NUTS ")
display(compare_single_df)


Trong bốn ngưỡng LOF 100%–90%–80%–70%, ngưỡng giữ lại 90% dữ liệu cho hiệu suất tốt nhất trên mô hình NUTS, đồng thời Metropolis cũng đạt kết quả tương đương. Vì vậy, trong các phân tích tiếp theo, em lựa chọn cấu hình LOF 90% + NUTS làm mô hình chính, còn Metropolis đóng vai trò đối chiếu để kiểm tra độ ổn định của suy luận Bayes.

### Ma trận nhầm lẫn cho 2 mô hình

In [ ]:
#  Hàm vẽ Confusion Matrix chung

import seaborn as sns
import matplotlib.pyplot as plt

def plot_conf_mat(cm, title="", class_labels=None):
    if class_labels is None:

        class_labels = [1, 2, 3, 4, 5, 6]

    plt.figure(figsize=(5, 4))
    sns.heatmap(
        cm,
        annot=True,
        fmt="d",
        cmap="Blues",
        xticklabels=class_labels,
        yticklabels=class_labels,
    )
    plt.xlabel("Predicted class")
    plt.ylabel("True class")
    plt.title(title)
    plt.tight_layout()
    plt.show()


In [ ]:
# Chọn ngưỡng theo NUTS
best_row = nuts_results_df.sort_values("accuracy_nuts", ascending=False).iloc[0]
best_r = float(best_row["retain_ratio"])
print(f"Ngưỡng LOF được chọn (theo NUTS): {best_r} – giữ lại {int(best_r*100)}% dữ liệu")

# Confusion matrix Metropolis
plot_conf_mat(
    mh_conf_mats[best_r],
    title=f"Metropolis (handmade) – Confusion Matrix – LOF {int(best_r*100)}%"
)

# Confusion matrix NUTS
plot_conf_mat(
    nuts_conf_mats[best_r],
    title=f"NUTS – Confusion Matrix – LOF {int(best_r*100)}%"
)


Hai ma trận nhầm lẫn này so sánh kết quả dự đoán trên tập test của Metropolis (trên) và NUTS (dưới) khi sử dụng ngưỡng LOF giữ lại 90% dữ liệu.

Đường chéo chính ở cả hai ma trận đều có giá trị cao, cho thấy mô hình phân loại đúng phần lớn các ca của cả 6 lớp bệnh.

Một số sai sót vẫn xuất hiện giữa các cặp lớp có biểu hiện lâm sàng tương đối giống nhau, nhưng số lượng rất nhỏ.

So sánh từng ô, NUTS cải thiện nhẹ ở vài lớp (ví dụ tăng số ca dự đoán đúng ở lớp 5–6), trong khi không làm giảm hiệu suất ở các lớp còn lại.
Nhìn chung, LOF 90% kết hợp với NUTS cho phân bố dự đoán cân đối giữa các lớp, hạn chế hiện tượng bỏ sót các lớp hiếm.

### Trace plot và phân phối hậu nghiệm


In [ ]:
# Trace & Posterior NUTS ===

import arviz as az
import matplotlib.pyplot as plt

# 1. Trace plot cho alpha
az.plot_trace(
    idata_nuts,
    var_names=["alpha"],
    compact=True
)
plt.suptitle("NUTS – Trace plot: alpha (6 lớp)", y=1.02)
plt.show()

# 2. Trace plot cho một vài beta
az.plot_trace(
    idata_nuts,
    var_names=["beta"],
    coords={
        "beta_dim_0": [0, 1, 2, 3],
        "beta_dim_1": [0, 1, 2, 3,4,5]
    },
    compact=True
)
plt.suptitle("NUTS – Trace plot: một số beta", y=1.02)
plt.show()

# 3. Posterior của alpha
az.plot_posterior(
    idata_nuts,
    var_names=["alpha"],
    hdi_prob=0.95
)
plt.suptitle("NUTS – Posterior alpha (HDI 95%)", y=1.02)
plt.show()

# 4. Posterior của beta
az.plot_posterior(
    idata_nuts,
    var_names=["beta"],
    coords={
        "beta_dim_0": [0, 1, 2,3,4,5],
        "beta_dim_1": [0, 1, 2]
    },
    hdi_prob=0.95
)
plt.suptitle("NUTS – Posterior một số beta (HDI 95%)", y=1.02)
plt.show()


Nhận xét về trace plot (alpha, beta):

Các đường trace của alpha (6 lớp bệnh) dao động quanh một giá trị trung tâm ổn định và đi lại đều trong toàn bộ chuỗi mẫu, không bị trôi dần lên/xuống. Điều này cho thấy chuỗi MCMC đã hội tụ và trộn tốt.

Trace của một số beta đại diện (các hệ số cho vài đặc trưng – vài lớp) cũng dao động ổn định quanh một mức trung bình.

Nhận xét về phân phối hậu nghiệm của một số beta:

Các posterior beta cho thấy mức độ và chiều ảnh hưởng của từng đặc trưng lên từng lớp bệnh:

Nhìn tổng thể, phần lớn các posterior beta tập trung trong khoảng giá trị vừa phải (không quá lớn về độ lớn tuyệt đối), phù hợp với kỳ vọng lâm sàng: mỗi đặc trưng đóng góp một phần vào quyết định chẩn đoán, và mô hình Bayes kết hợp chúng lại một cách “mềm” thay vì phụ thuộc vào một vài đặc trưng cực đoan.

# PHẦN 6 - TỔNG QUAN

Bài toán chẩn đoán 6 bệnh trong nhóm Erythemato-Squamous bằng mô hình **Bayesian Multinomial Logistic Regression** đã được triển khai với hai phương pháp suy luận:

* **Metropolis–Hastings (handmade)**
* **NUTS (No-U-Turn Sampler) — PyMC**

Song song, dữ liệu được xử lý ngoại lai bằng thuật toán **Local Outlier Factor (LOF)** ở bốn ngưỡng:

* 100% dữ liệu (không lọc)
* 90% dữ liệu (lọc 10% cao nhất)
* 80%
* 70%

**Kết quả quan trọng nhất:**

* Cả hai mô hình đều đạt **độ chính xác rất cao (95–98%)** trên test set.
* Mô hình NUTS có **log loss tốt hơn**, ổn định hơn → cho thấy **độ tin cậy cao hơn** trong xác suất dự đoán.
* Ngưỡng LOF **90%** cho hiệu suất tổng thể tốt nhất (Accuracy, Macro-F1, LogLoss, AUC).

Điều này cho thấy việc xử lý ngoại lai có ý nghĩa rõ rệt trong việc giúp mô hình học được ranh giới quyết định ổn định hơn.


## Hạn chế của dữ liệu & phương pháp

### **(1) Bộ dữ liệu rất nhỏ: chỉ 366 mẫu**

* Với 34 đặc trưng + 6 lớp → nguy cơ **overfitting cao**.
* Các mẫu ngoại lai có thể thật sự là bệnh nhân "đặc biệt" → việc lọc bỏ cần cẩn trọng trong ứng dụng thực tế.
* Không đảm bảo phân phối dữ liệu đại diện cho dân số rộng.

### **(2) Một số đặc trưng không thật sự phân biệt rõ ràng**

* Một số β có HDI chứa 0 → đặc trưng đó "không đóng góp nhiều", hoặc dữ liệu chưa đủ mạnh.
* Điều này cho thấy **mô hình chưa tận dụng hết 34 đặc trưng**, có thể do collinearity hoặc thông tin nhiễu.

### **(3) Mô hình đa thức logistic là tuyến tính**

* Không nắm bắt được mối quan hệ phi tuyến giữa đặc trưng.
* Các mô hình như RandomForest hoặc XGBoost có thể học tốt hơn — nhưng lại thiếu phân phối xác suất dạng Bayes.

### **(4) ROC-AUC gần 1.0 nhưng dễ gây "ảo tưởng hiệu suất"**

* Vì dữ liệu nhỏ, sạch → mô hình dễ đạt AUC rất cao.
* Trong thực tế, độ nhiễu cao hơn → AUC có thể giảm.

## Kết luận thực tế

Kết quả cho thấy rằng:

* **Xử lý ngoại lai bằng LOF (đặc biệt ở 90%) cải thiện rõ rệt độ ổn định và chất lượng mô hình.**
* **NUTS là phương pháp suy luận đáng tin cậy nhất**, hội tụ nhanh, posterior đẹp, log loss thấp.
* Mô hình Bayes không chỉ dự đoán tốt mà còn **diễn giải được**, rất phù hợp cho lĩnh vực y khoa.
* Tuy nhiên, **hạn chế về kích thước dữ liệu** khiến các kết luận chỉ mang tính minh hoạ, không thể áp dụng trực tiếp vào lâm sàng.

→ **Nếu bộ dữ liệu lớn hơn (tối thiểu 5.000–10.000 mẫu), mô hình Bayes sẽ cho kết quả ổn định và có giá trị sử dụng lâm sàng cao hơn.**



In [ ]:
from sklearn.metrics import roc_curve, auc
import matplotlib.pyplot as plt
import numpy as np

y_true = y_test           # nhãn thật (đã có)
y_prob = proba_nuts       # xác suất từ posterior NUTS (đã có)
n_classes = y_prob.shape[1]

plt.figure(figsize=(8,6))
macro_auc_list = []

for c in range(n_classes):
    # chuyển y_true về dạng nhị phân ONE-vs-REST
    y_bin = (y_true == c).astype(int)

    if y_bin.sum() == 0:
        continue

    fpr, tpr, _ = roc_curve(y_bin, y_prob[:, c])
    auc_val = auc(fpr, tpr)
    macro_auc_list.append(auc_val)

    plt.plot(fpr, tpr, lw=2, label=f"Class {c+1} AUC = {auc_val:.3f}")

# baseline
plt.plot([0,1],[0,1],"k--",lw=1)

plt.xlabel("False Positive Rate")
plt.ylabel("True Positive Rate")
plt.title(f"NUTS – ROC Curves (single split)\nMacro AUC = {np.mean(macro_auc_list):.4f}")
plt.legend(loc="lower right")
plt.grid(alpha=0.3)
plt.show()


In [ ]:
import numpy as np
import pandas as pd
from sklearn.model_selection import KFold
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import accuracy_score, f1_score, log_loss, roc_auc_score

# 1. Chọn ngưỡng LOF đẹp nhất
best_ratio = 0.9

# 2. Lấy dataset đã xử lý LOF tương ứng ngưỡng đẹp nhất
df_best = datasets_lof[best_ratio].copy()

X = df_best.drop(columns=["class"]).values
y = df_best["class"].values
classes = np.unique(y)

# 3. Khởi tạo 5-fold
kf = KFold(n_splits=5, shuffle=True, random_state=42)

nuts_cv_results = []
fold = 1

for train_idx, test_idx in kf.split(X, y):
    print(f"\nFold {fold}/5")

    X_train, X_test = X[train_idx], X[test_idx]
    y_train, y_test = y[train_idx], y[test_idx]

    # 3.1. Scale dữ liệu
    scaler = StandardScaler()
    X_train_s = scaler.fit_transform(X_train)
    X_test_s  = scaler.transform(X_test)

    # 3.2. Fit mô hình NUTS trên fold này
    model_nuts, idata_nuts = fit_bayes_multilogit_nuts(
        X_train_s, y_train,
        draws=800,
        tune=800,
        target_accept=0.9,
        random_seed=fold
    )

    # 3.3. Dự đoán xác suất theo posterior mean
    proba_nuts = predict_bayes_multilogit_mean(idata_nuts, X_test_s)
    y_pred = proba_nuts.argmax(axis=1)

    # 3.4. Tính các metric
    acc   = accuracy_score(y_test, y_pred)
    macro_f1 = f1_score(y_test, y_pred, average="macro")
    ll    = log_loss(y_test, proba_nuts)
    auc_ovr = roc_auc_score(pd.get_dummies(y_test), proba_nuts, multi_class="ovr")

    nuts_cv_results.append([acc, macro_f1, ll, auc_ovr])

    print(f"  Accuracy   : {acc:.4f}")
    print(f"  Macro F1   : {macro_f1:.4f}")
    print(f"  Log loss   : {ll:.4f}")
    print(f"  AUC (OvR)  : {auc_ovr:.4f}")

    fold += 1

# 4. Tổng hợp kết quả 5 fold
cols = ["Accuracy", "Macro_F1", "LogLoss", "AUC_OvR"]
nuts_cv_df = pd.DataFrame(nuts_cv_results, columns=cols)

print("\n NUTS – 5-fold CV LOF 90% ")
display(nuts_cv_df)
print("\nTrung bình 5 fold:")
display(nuts_cv_df.mean().to_frame("NUTS_mean"))


Các chỉ số từ kiểm định chéo 5 lần cho thấy mô hình NUTS đạt hiệu suất rất cao và ổn định (Accuracy ≈ 96.66%, Macro-F1 ≈ 96.25%). Giá trị log-loss thấp (≈ 0.11) chứng tỏ mô hình đưa ra xác suất dự đoán tự tin. Chỉ số AUC OvR ≈ 0.9985 cho thấy mô hình gần như phân biệt hoàn hảo giữa 6 nhóm bệnh lý, không bị lệch lớp và có khả năng tổng quát hóa mạnh mẽ trên các phân chia dữ liệu khác nhau.

In [ ]:
!pip uninstall -y ipywidgets